# Práctica 3 — Inferencia y contratos

Daniel Alonso · Operación de Modelos · 14/10/2026

He completado la práctica de la clase 1 para fijar el contrato que utilizo en el módulo local de la clase 2. Aquí compruebo datos, validación y orden; el CLI está en el starter.

## Ejecución

Ejecuto las celdas en orden desde esta carpeta. Los datos son la muestra docente de cinco vinos, procedente de Red Wine Quality (UCI/Kaggle), sin la etiqueta `quality`.

Distingo tres elementos: los datos de entrenamiento sirvieron para ajustar el estimador; el artefacto `.joblib` guarda ese estimador y sus metadatos; los datos nuevos solo se utilizan para inferencia. No reentreno ni ajusto un escalador con estas cinco filas.

In [1]:
from dataclasses import dataclass
from typing import Literal
import csv
from pathlib import Path

from pydantic import BaseModel, ConfigDict, Field, ValidationError

# Mantengo la ruta relativa del material original de la práctica.
DATA_PATH = Path("../../../../../../assets/03-wine-quality/inference_samples.csv")
assert DATA_PATH.is_file(), "Ejecuta el notebook desde su carpeta."
print("CSV disponible:", DATA_PATH.is_file())

CSV disponible: True


## 1. Leer los datos nuevos

### Práctica A — Inspección del CSV

Completa la celda siguiente para:

- Leer el CSV con `csv.DictReader`.
- Imprimir el número de filas.
- Imprimir las columnas.
- Guardar la primera fila en una variable llamada `row`.

In [2]:
# Leo los datos sin convertir todavía las cadenas numéricas.
with DATA_PATH.open(newline="", encoding="utf-8") as source:
    rows = list(csv.DictReader(source))
assert len(rows) == 5
row = rows[0]
print("Número de filas:", len(rows))
print("Columnas:", list(row))
print("Primera fila:", row)

Número de filas: 5
Columnas: ['sample_id', 'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol']
Primera fila: {'sample_id': 'red-001', 'fixed_acidity': '7.4', 'volatile_acidity': '0.7', 'citric_acid': '0.0', 'residual_sugar': '1.9', 'chlorides': '0.076', 'free_sulfur_dioxide': '11.0', 'total_sulfur_dioxide': '34.0', 'density': '0.9978', 'ph': '3.51', 'sulphates': '0.56', 'alcohol': '9.4'}


### Preguntas

1. ¿Qué campo identifica la muestra pero no es una característica del modelo?
2. ¿Cuántas características numéricas hay?
3. ¿Qué columna extra añadirías para probar el contrato?
4. ¿Qué valor de `ph` sería claramente inválido?

**Mis respuestas:** `sample_id` identifica la muestra y no entra en el modelo. Hay 11 características numéricas. Añadiría `unexpected_field` para comprobar el rechazo. Un `ph=99` incumple el rango 2.5–4.5.

## 2. Construir el contrato con Pydantic

| Campo | Tipo validado | Unidad asumida | Límite aceptado |
| --- | --- | --- | --- |
| `fixed_acidity` | `float` | g/L de ácido tartárico | 0–20 |
| `volatile_acidity` | `float` | g/L de ácido acético | 0–2 |
| `citric_acid` | `float` | g/L | 0–2 |
| `residual_sugar` | `float` | g/L | 0–20 |
| `chlorides` | `float` | g/L de NaCl | 0–1 |
| `free_sulfur_dioxide` | `float` | mg/L | 0–100 |
| `total_sulfur_dioxide` | `float` | mg/L | 0–300 |
| `density` | `float` | g/cm³ | 0.98–1.01 |
| `ph` | `float` | adimensional | 2.5–4.5 |
| `sulphates` | `float` | g/L de K₂SO₄ | 0–3 |
| `alcohol` | `float` | % vol. | 5–20 |

Asumo las unidades habituales del dataset; el CSV no permite verificarlas automáticamente. Mantengo los rangos orientativos del profesor como contrato inclusivo.


In [3]:
class WineQualityRequestPydantic(BaseModel):
    # Rechazo columnas extra y números no finitos; convierto el texto del CSV.
    model_config = ConfigDict(extra="forbid", allow_inf_nan=False)

    fixed_acidity: float = Field(ge=0, le=20)
    volatile_acidity: float = Field(ge=0, le=2)
    citric_acid: float = Field(ge=0, le=2)
    residual_sugar: float = Field(ge=0, le=20)
    chlorides: float = Field(ge=0, le=1)
    free_sulfur_dioxide: float = Field(ge=0, le=100)
    total_sulfur_dioxide: float = Field(ge=0, le=300)
    density: float = Field(ge=0.98, le=1.01)
    ph: float = Field(ge=2.5, le=4.5)
    sulphates: float = Field(ge=0, le=3)
    alcohol: float = Field(ge=5, le=20)

# Separo solo el identificador: no oculto otras columnas al contrato.
model_row = {name: value for name, value in row.items() if name != "sample_id"}
request = WineQualityRequestPydantic.model_validate(model_row)
print(request.model_dump())
assert isinstance(request.alcohol, float)

{'fixed_acidity': 7.4, 'volatile_acidity': 0.7, 'citric_acid': 0.0, 'residual_sugar': 1.9, 'chlorides': 0.076, 'free_sulfur_dioxide': 11.0, 'total_sulfur_dioxide': 34.0, 'density': 0.9978, 'ph': 3.51, 'sulphates': 0.56, 'alcohol': 9.4}


### Práctica B — Validación

Completa una prueba para cada caso:

- Una fila válida.
- Una columna `unexpected_field`.
- Un `ph` fuera de rango.

Antes de ejecutar, escribe qué error esperas.

**Antes de ejecutar:** espero que la fila válida se convierta a floats, que la columna extra genere `extra_forbidden` y que `ph=99` genere `less_than_equal`.

In [4]:
cases = {
    "fila válida": model_row,
    "columna extra": {**model_row, "unexpected_field": "not allowed"},
    "ph fuera de rango": {**model_row, "ph": "99"},
}
for name, values in cases.items():
    try:
        WineQualityRequestPydantic.model_validate(values)
    except ValidationError as error:
        # Espero extra_forbidden para la columna y less_than_equal para ph.
        assert name != "fila válida"
        print(name, "->", error.errors()[0]["type"])
    else:
        assert name == "fila válida"
        print(name, "-> aceptada")

fila válida -> aceptada
columna extra -> extra_forbidden
ph fuera de rango -> less_than_equal


## 3. Comparar con `dataclass`

Construye la misma estructura con `dataclass`. No añadas todavía validación manual: observa qué garantiza Python y qué no garantiza.

In [5]:
@dataclass(frozen=True, slots=True)
class WineQualityRequestDataclass:
    fixed_acidity: float
    volatile_acidity: float
    citric_acid: float
    residual_sugar: float
    chlorides: float
    free_sulfur_dioxide: float
    total_sulfur_dioxide: float
    density: float
    ph: float
    sulphates: float
    alcohol: float

valid_dataclass = WineQualityRequestDataclass(**request.model_dump())
invalid_dataclass = WineQualityRequestDataclass(**{**request.model_dump(), "ph": 99.0})
# Las anotaciones de una dataclass no comprueban el rango en runtime.
print("ph válido:", valid_dataclass.ph)
print("ph=99 aceptado por dataclass:", invalid_dataclass.ph)
assert invalid_dataclass.ph == 99.0

ph válido: 3.51
ph=99 aceptado por dataclass: 99.0


### Decisión de diseño

Completa la frase:

> Usaría Pydantic en __________ porque __________. Usaría `dataclass` en __________ porque __________.

Usaría Pydantic en la frontera de entrada y salida porque valida datos en runtime. Usaría `dataclass` para el vector interno ya validado porque conserva una estructura sencilla y puede ser inmutable. `frozen=True` impide reasignar atributos, pero no valida valores.

## 4. Definir el orden del vector

El modelo no recibe un diccionario. Recibe una lista numérica con un orden estable.

In [6]:
# Centralizo el orden; no dependo del orden de las columnas de cada CSV.
FEATURE_NAMES = (
    "fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar",
    "chlorides", "free_sulfur_dioxide", "total_sulfur_dioxide", "density",
    "ph", "sulphates", "alcohol",
)
vector = [getattr(request, name) for name in FEATURE_NAMES]
assert len(vector) == 11
assert "sample_id" not in FEATURE_NAMES
assert vector[7] == request.density and vector[10] == request.alcohol
print("Orden:", FEATURE_NAMES)
print("Vector:", vector)

Orden: ('fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol')
Vector: [7.4, 0.7, 0.0, 1.9, 0.076, 11.0, 34.0, 0.9978, 3.51, 0.56, 9.4]


### Práctica C — Error de orden

Intercambia mentalmente `density` y `alcohol`. Responde:

- ¿El vector sigue teniendo 11 valores?
- ¿El clasificador detectaría necesariamente el error?
- ¿Cómo evitarías que cada script definiera un orden distinto?

**Mi respuesta:** el vector seguiría teniendo 11 valores, pero el clasificador no tiene por qué detectar el intercambio: recibe números en posiciones incorrectas. Para evitarlo, defino un único `FEATURE_NAMES` y comparo esa tupla con los metadatos del artefacto antes de inferir.

## 5. Contrato de salida

Diseña la estructura que devolverá el módulo. Debe permitir responder:

- ¿Qué muestra se ha procesado?
- ¿Qué categoría se ha predicho?
- ¿Con qué confianza?
- ¿Qué versión de modelo y preprocesado se utilizaron?

In [7]:
class WineQualityPredictionPydantic(BaseModel):
    model_config = ConfigDict(extra="forbid", allow_inf_nan=False)

    sample_id: str = Field(min_length=1)
    quality_band: Literal["needs_review", "acceptable", "excellent"]
    confidence: float = Field(ge=0, le=1)
    model_version: str = Field(min_length=1)
    preprocessing_version: str = Field(min_length=1)

# Ejemplo de estructura, no una ejecución del clasificador en esta celda.
example = WineQualityPredictionPydantic(
    sample_id="red-001", quality_band="needs_review", confidence=0.8,
    model_version="wine-quality-rf-v1", preprocessing_version="wine-red-features-v1",
)
print(example.model_dump())

{'sample_id': 'red-001', 'quality_band': 'needs_review', 'confidence': 0.8, 'model_version': 'wine-quality-rf-v1', 'preprocessing_version': 'wine-red-features-v1'}


## 6. Recorrido de la muestra y plan de implementación

`CSV → contrato → preprocesado → modelo cargado → predicción → CSV de salida`

1. En `contracts.py` valido tipos, rangos y campos permitidos, además de la salida.
2. En `preprocess.py` preparo once valores en `FEATURE_NAMES`, sin `sample_id`, y con versión.
3. En `inference.py` compruebo el artefacto, cargo el estimador y uso `predict`/`predict_proba`.
4. En `predict_file.py` valido todas las filas, cargo una vez el modelo, infiero y escribo al final.

Mis tres criterios de aceptación son: cinco filas válidas generan cinco salidas trazables; una columna extra o una fila inválida falla antes de inferir y no deja salida parcial; un artefacto con otro orden de características se rechaza antes de predecir.

## Entrega

El lienzo completo con unidades, límites y respuestas está en [LIENZO.md](LIENZO.md). La implementación y los tests están en el starter de la clase 2. La explicación y los resultados reales del CLI están en [DESARROLLO.md](../../../../../../DESARROLLO.md).

La confianza es la probabilidad de la categoría predicha según el clasificador; no presupongo que esté calibrada ni puedo medir accuracy con este CSV sin etiquetas.

La exportación con tabla, respuestas, casos, vector, salida y plan está en [01-inferencia-y-contratos.html](../../../../../../entrega/01-inferencia-y-contratos.html). Es la alternativa a una captura permitida por el enunciado.
